# 🎓 Student Performance Predictor
### Minor Project II — CSE (Data Science) | RGPV Bhopal | 2024-25

**Team:** NAME-1 | NAME-2 | NAME-3  
**Guide:** [Guide Name] | **Co-Guide:** [Co-Guide Name]

---
## Notebook Overview
This notebook covers the **complete ML pipeline**:
1. Library imports & configuration
2. Dataset loading & overview
3. Exploratory Data Analysis (EDA)
4. Data preprocessing
5. Model training (6 models)
6. Model evaluation & comparison
7. Feature importance analysis
8. Saving trained models
9. Prediction function

## 1. Library Imports & Configuration

In [ ]:
# ── Standard Libraries ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

# ── Machine Learning ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score
)

print('✅ All libraries imported successfully!')
print(f'   pandas    : {pd.__version__}')
print(f'   numpy     : {np.__version__}')
import sklearn; print(f'   sklearn   : {sklearn.__version__}')

## 2. Dataset Loading & Overview

In [ ]:
# ── Load Dataset ────────────────────────────────────────────────────
df = pd.read_csv('student_data.csv')

print(f'Dataset Shape : {df.shape}')
print(f'Rows          : {df.shape[0]}')
print(f'Columns       : {df.shape[1]}')
print('\n--- First 5 Rows ---')
df.head()

In [ ]:
# ── Dataset Info ─────────────────────────────────────────────────────
print('--- Data Types & Non-null Counts ---')
df.info()

In [ ]:
# ── Statistical Summary ──────────────────────────────────────────────
print('--- Statistical Summary ---')
df.describe().round(2)

In [ ]:
# ── Target Variable Distribution ─────────────────────────────────────
print('Grade Distribution:')
print(df['grade'].value_counts())
print('\nPass/Fail Distribution:')
print(df['result'].value_counts())
print('\nMissing Values:')
print(df.isnull().sum())

## 3. Exploratory Data Analysis (EDA)
### 3.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Grade distribution
grade_order = ['O', 'A+', 'A', 'B+', 'B', 'C', 'F']
grade_counts = df['grade'].value_counts().reindex(grade_order, fill_value=0)
colors_grade = ['#1abc9c','#27ae60','#2980b9','#8e44ad','#f39c12','#e67e22','#e74c3c']
axes[0].bar(grade_counts.index, grade_counts.values, color=colors_grade, edgecolor='white', linewidth=1.5)
axes[0].set_title('Grade Distribution', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Grade'); axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(grade_counts.items()):
    axes[0].text(i, val + 1, str(val), ha='center', fontweight='bold')

# Pass/Fail pie
result_counts = df['result'].value_counts()
axes[1].pie(result_counts.values, labels=result_counts.index,
            colors=['#27ae60','#e74c3c'], autopct='%1.1f%%',
            startangle=140, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Pass / Fail Ratio', fontweight='bold', fontsize=14)

# Final score histogram
axes[2].hist(df['final_score'], bins=25, color='#5c67f2', edgecolor='white', linewidth=0.8, alpha=0.85)
axes[2].axvline(df['final_score'].mean(), color='#e74c3c', linestyle='--', linewidth=2,
                label=f'Mean: {df["final_score"].mean():.1f}')
axes[2].axvline(40, color='#f39c12', linestyle=':', linewidth=2, label='Pass Line (40)')
axes[2].set_title('Final Score Distribution', fontweight='bold', fontsize=14)
axes[2].set_xlabel('Score'); axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.suptitle('Target Variable Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.show()

### 3.2 Feature Distributions

In [ ]:
num_features = ['attendance', 'prev_marks', 'study_hours', 'assignments', 'internals', 'absences']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
palette = ['#5c67f2','#27ae60','#e67e22','#8e44ad','#2980b9','#e74c3c']

for ax, feat, color in zip(axes.flat, num_features, palette):
    ax.hist(df[feat], bins=20, color=color, edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.axvline(df[feat].mean(), color='black', linestyle='--', linewidth=1.8,
               label=f'Mean: {df[feat].mean():.1f}')
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.show()

### 3.3 Correlation Heatmap

In [ ]:
FEATURES = ['attendance','prev_marks','study_hours','assignments',
            'internals','extracurricular','parental_edu','internet',
            'health','absences']

corr = df[FEATURES + ['final_score']].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\nTop correlations with final_score:')
print(corr['final_score'].drop('final_score').sort_values(ascending=False))

### 3.4 Feature vs Final Score (Scatter Plots)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, feat in zip(axes.flat, num_features):
    sc = ax.scatter(df[feat], df['final_score'],
                    c=df['final_score'], cmap='RdYlGn',
                    alpha=0.5, edgecolors='none', s=20)
    z = np.polyfit(df[feat], df['final_score'], 1)
    p = np.poly1d(z)
    xs = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(xs, p(xs), 'r--', linewidth=2, alpha=0.8)
    r = df[feat].corr(df['final_score'])
    ax.set_title(f'{feat.replace("_"," ").title()}  (r={r:.3f})', fontweight='bold')
    ax.set_xlabel(feat); ax.set_ylabel('Final Score')

plt.tight_layout()
plt.suptitle('Feature vs Final Score', fontsize=16, fontweight='bold', y=1.02)
plt.show()

### 3.5 Boxplot: Score by Grade

In [ ]:
grade_order = [g for g in ['O','A+','A','B+','B','C','F'] if g in df['grade'].values]
colors_grade = ['#1abc9c','#27ae60','#2980b9','#8e44ad','#f39c12','#e67e22','#e74c3c']

plt.figure(figsize=(11, 5))
data_by_grade = [df[df['grade']==g]['final_score'].values for g in grade_order]
bp = plt.boxplot(data_by_grade, labels=grade_order, patch_artist=True,
                 notch=False, medianprops={'color':'black','linewidth':2.5})
for patch, color in zip(bp['boxes'], colors_grade[:len(grade_order)]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
plt.title('Final Score Distribution by Grade', fontsize=15, fontweight='bold')
plt.xlabel('Grade', fontsize=12); plt.ylabel('Final Score', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
FEATURES = ['attendance','prev_marks','study_hours','assignments',
            'internals','extracurricular','parental_edu','internet',
            'health','absences']

X = df[FEATURES]
y_score  = df['final_score']                         # Regression target
y_pass   = (df['final_score'] >= 40).astype(int)     # Classification: PASS=1, FAIL=0
y_grade  = df['grade']                               # Multi-class grade

# ── Scaling ──────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES)

# ── Train-Test Split (80/20) ─────────────────────────────────────────
Xtr, Xte, ytr_s, yte_s = train_test_split(X_scaled, y_score, test_size=0.2, random_state=42)
_,   _,   ytr_p, yte_p = train_test_split(X_scaled, y_pass,  test_size=0.2, random_state=42)
_,   _,   ytr_g, yte_g = train_test_split(X_scaled, y_grade, test_size=0.2, random_state=42)

print(f'Training samples : {len(Xtr)}')
print(f'Testing  samples : {len(Xte)}')
print(f'Features         : {list(FEATURES)}')
print(f'\nClass balance (y_pass):')
print(pd.Series(ytr_p).value_counts().rename({0:'FAIL',1:'PASS'}))

In [ ]:
# ── Visualize Scaled Features ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([X[f].values for f in FEATURES], labels=FEATURES, vert=False, patch_artist=True,
                boxprops={'facecolor':'#5c67f2','alpha':0.6})
axes[0].set_title('Before Scaling', fontweight='bold')
axes[0].set_xlabel('Value')

axes[1].boxplot([X_scaled[f].values for f in FEATURES], labels=FEATURES, vert=False, patch_artist=True,
                boxprops={'facecolor':'#27ae60','alpha':0.6})
axes[1].set_title('After StandardScaler', fontweight='bold')
axes[1].set_xlabel('Standardized Value')

plt.suptitle('Feature Scaling Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Model Training

We train **6 models** — 3 for score regression, 3 for pass/fail classification:

| Model | Task | Type |
|-------|------|------|
| Linear Regression | Score prediction | Regression |
| Random Forest Regressor | Score prediction | Regression |
| Gradient Boosting Regressor | Score prediction | Regression |
| Logistic Regression | Pass/Fail | Classification |
| Random Forest Classifier | Pass/Fail | Classification |
| SVM (RBF Kernel) | Pass/Fail | Classification |

### 5.1 Regression Models — Score Prediction

In [ ]:
results_reg = {}

# ── 1. Linear Regression ─────────────────────────────────────────────
print('Training Linear Regression...')
lin_reg = LinearRegression()
lin_reg.fit(Xtr, ytr_s)
pred_lr   = np.clip(lin_reg.predict(Xte), 0, 100)
rmse_lr   = np.sqrt(mean_squared_error(yte_s, pred_lr))
r2_lr     = r2_score(yte_s, pred_lr)
results_reg['Linear Regression'] = {'RMSE': round(rmse_lr,4), 'R2': round(r2_lr,4)}
print(f'   RMSE = {rmse_lr:.4f}  |  R2 = {r2_lr:.4f}')

# ── 2. Random Forest Regressor ───────────────────────────────────────
print('Training Random Forest Regressor...')
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=5,
                                min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_reg.fit(Xtr, ytr_s)
pred_rfr  = np.clip(rf_reg.predict(Xte), 0, 100)
rmse_rfr  = np.sqrt(mean_squared_error(yte_s, pred_rfr))
r2_rfr    = r2_score(yte_s, pred_rfr)
results_reg['Random Forest Reg'] = {'RMSE': round(rmse_rfr,4), 'R2': round(r2_rfr,4)}
print(f'   RMSE = {rmse_rfr:.4f}  |  R2 = {r2_rfr:.4f}')

# ── 3. Gradient Boosting ─────────────────────────────────────────────
print('Training Gradient Boosting...')
gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                    max_depth=4, random_state=42)
gb_reg.fit(Xtr, ytr_s)
pred_gbr  = np.clip(gb_reg.predict(Xte), 0, 100)
rmse_gbr  = np.sqrt(mean_squared_error(yte_s, pred_gbr))
r2_gbr    = r2_score(yte_s, pred_gbr)
results_reg['Gradient Boosting'] = {'RMSE': round(rmse_gbr,4), 'R2': round(r2_gbr,4)}
print(f'   RMSE = {rmse_gbr:.4f}  |  R2 = {r2_gbr:.4f}')

print('\n✅ Regression models trained!')

### 5.2 Classification Models — Pass/Fail Prediction

In [ ]:
results_clf = {}

# ── 4. Logistic Regression ───────────────────────────────────────────
print('Training Logistic Regression...')
log_reg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
log_reg.fit(Xtr, ytr_p)
pred_log  = log_reg.predict(Xte)
acc_log   = accuracy_score(yte_p, pred_log)
results_clf['Logistic Regression'] = {'Accuracy': round(acc_log*100, 2)}
print(f'   Accuracy = {acc_log*100:.2f}%')

# ── 5. Random Forest Classifier ──────────────────────────────────────
print('Training Random Forest Classifier...')
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                  min_samples_split=5, random_state=42, n_jobs=-1)
rf_clf.fit(Xtr, ytr_p)
pred_rfc  = rf_clf.predict(Xte)
acc_rfc   = accuracy_score(yte_p, pred_rfc)
results_clf['Random Forest Clf'] = {'Accuracy': round(acc_rfc*100, 2)}
print(f'   Accuracy = {acc_rfc*100:.2f}%')

# ── 6. SVM ────────────────────────────────────────────────────────────
print('Training SVM (RBF Kernel)...')
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale',
                 probability=True, random_state=42)
svm_model.fit(Xtr, ytr_p)
pred_svm  = svm_model.predict(Xte)
acc_svm   = accuracy_score(yte_p, pred_svm)
results_clf['SVM (RBF)'] = {'Accuracy': round(acc_svm*100, 2)}
print(f'   Accuracy = {acc_svm*100:.2f}%')

print('\n✅ Classification models trained!')

## 6. Model Evaluation & Comparison

### 6.1 Regression Results

In [ ]:
reg_df = pd.DataFrame(results_reg).T.reset_index()
reg_df.columns = ['Model', 'RMSE', 'R2 Score']
reg_df['RMSE']     = reg_df['RMSE'].astype(float)
reg_df['R2 Score'] = reg_df['R2 Score'].astype(float)

print('=== Regression Models Comparison ===')
print(reg_df.to_string(index=False))
print(f'\nBest R2 : {reg_df.loc[reg_df["R2 Score"].idxmax(), "Model"]}')
print(f'Best RMSE: {reg_df.loc[reg_df["RMSE"].idxmin(), "Model"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_m = ['#5c67f2','#27ae60','#e67e22']

# RMSE
bars0 = axes[0].bar(reg_df['Model'], reg_df['RMSE'], color=colors_m,
                     edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('RMSE Comparison\n(Lower = Better)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('RMSE')
for bar, val in zip(bars0, reg_df['RMSE']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.3f}', ha='center', fontweight='bold')

# R2
bars1 = axes[1].bar(reg_df['Model'], reg_df['R2 Score'], color=colors_m,
                     edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('R\u00b2 Score Comparison\n(Higher = Better)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('R\u00b2 Score'); axes[1].set_ylim(0, 1)
for bar, val in zip(bars1, reg_df['R2 Score']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.suptitle('Regression Model Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Actual vs Predicted (best model)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
preds   = [pred_lr, pred_rfr, pred_gbr]
labels  = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
colors  = ['#5c67f2','#27ae60','#e67e22']

for ax, pred, label, color in zip(axes, preds, labels, colors):
    ax.scatter(yte_s, pred, alpha=0.5, color=color, edgecolors='none', s=30)
    lim = [0, 100]
    ax.plot(lim, lim, 'r--', linewidth=2, label='Perfect Prediction')
    r2 = r2_score(yte_s, pred)
    ax.set_title(f'{label}\nR\u00b2 = {r2:.4f}', fontweight='bold')
    ax.set_xlabel('Actual Score'); ax.set_ylabel('Predicted Score')
    ax.set_xlim(0,100); ax.set_ylim(0,100)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Actual vs Predicted Score', fontsize=15, fontweight='bold', y=1.02)
plt.show()

### 6.2 Classification Results

In [ ]:
clf_df = pd.DataFrame(results_clf).T.reset_index()
clf_df.columns = ['Model', 'Accuracy %']
print('=== Classification Models Comparison ===')
print(clf_df.to_string(index=False))

# Detailed report for best model (RF Classifier)
print('\n=== Detailed Report: Random Forest Classifier ===')
print(classification_report(yte_p, pred_rfc, target_names=['FAIL', 'PASS']))

In [ ]:
# Confusion Matrices for all classifiers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
all_preds  = [pred_log, pred_rfc, pred_svm]
clf_labels = ['Logistic Regression', 'Random Forest', 'SVM (RBF)']
cmaps      = ['Blues', 'Greens', 'Oranges']

for ax, pred, label, cmap in zip(axes, all_preds, clf_labels, cmaps):
    cm = confusion_matrix(yte_p, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['FAIL','PASS'])
    disp.plot(ax=ax, cmap=cmap, colorbar=False)
    acc = accuracy_score(yte_p, pred)
    ax.set_title(f'{label}\nAccuracy: {acc*100:.2f}%', fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### 6.3 Cross-Validation

In [ ]:
print('=== 5-Fold Cross Validation ===')
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model, y_target, scoring in [
    ('Linear Regression',   LinearRegression(),                         y_score, 'r2'),
    ('RF Regressor',        RandomForestRegressor(n_estimators=50, random_state=42), y_score, 'r2'),
    ('Gradient Boosting',   GradientBoostingRegressor(n_estimators=50, random_state=42), y_score, 'r2'),
    ('Logistic Regression', LogisticRegression(max_iter=500,random_state=42), y_pass, 'accuracy'),
    ('RF Classifier',       RandomForestClassifier(n_estimators=50,random_state=42), y_pass, 'accuracy'),
    ('SVM',                 SVC(kernel="rbf",random_state=42), y_pass, 'accuracy'),
]:
    scores = cross_val_score(model, X_scaled, y_target, cv=kf, scoring=scoring)
    cv_results[name] = {'Mean': round(scores.mean(),4), 'Std': round(scores.std(),4), 'Metric': scoring}
    print(f'{name:<25}: {scores.mean():.4f} +/- {scores.std():.4f}  ({scoring})')

cv_df = pd.DataFrame(cv_results).T.reset_index()
cv_df.columns = ['Model','Mean','Std','Metric']
print()
cv_df

## 7. Feature Importance Analysis

In [ ]:
# ── Random Forest Feature Importance ────────────────────────────────
fi = pd.Series(rf_reg.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi)))
bars = axes[0].barh(fi.index[::-1], fi.values[::-1], color=colors_fi[::-1],
                     edgecolor='white', linewidth=1, height=0.6)
for bar, val in zip(bars, fi.values[::-1]):
    axes[0].text(val+0.002, bar.get_y()+bar.get_height()/2,
                 f'{val:.3f}', va='center', fontweight='bold', fontsize=10)
axes[0].set_title('Feature Importance\n(Random Forest Regressor)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Importance Score')

# Pie chart
axes[1].pie(fi.values, labels=fi.index, autopct='%1.1f%%',
             startangle=140, colors=plt.cm.Set3(np.linspace(0, 1, len(fi))),
             wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Feature Importance Share', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.suptitle('Feature Importance Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.show()

print('\nFeature Importance Rankings:')
for rank, (feat, imp) in enumerate(fi.items(), 1):
    bar_fill = '#' + '2ecc71' if imp > 0.15 else ('f39c12' if imp > 0.08 else 'e74c3c')
    print(f'{rank:2}. {feat:<18} | {imp:.4f} | {"*"*int(imp*100)}')

## 8. Save Trained Models

In [ ]:
os.makedirs('models', exist_ok=True)

model_map = {
    'models/scaler.pkl':      scaler,
    'models/lin_reg.pkl':     lin_reg,
    'models/rf_reg.pkl':      rf_reg,
    'models/gb_reg.pkl':      gb_reg,
    'models/log_reg.pkl':     log_reg,
    'models/rf_clf.pkl':      rf_clf,
    'models/svm.pkl':         svm_model,
    'models/feature_importance.pkl': fi,
    'models/confusion_matrix.pkl':   confusion_matrix(yte_p, pred_rfc),
    'models/clf_report.pkl':         classification_report(yte_p, pred_rfc,
                                         target_names=['FAIL','PASS'], output_dict=True),
}

for path, obj in model_map.items():
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  Saved: {path}')

print('\n✅ All models saved to ./models/')

## 9. Prediction Function

In [ ]:
def score_to_grade(score):
    """Convert numeric score to letter grade (RGPV scale)"""
    if score >= 90: return 'O'
    elif score >= 80: return 'A+'
    elif score >= 70: return 'A'
    elif score >= 60: return 'B+'
    elif score >= 50: return 'B'
    elif score >= 40: return 'C'
    else: return 'F'

def get_risk_level(score):
    """Determine academic risk level"""
    if score < 45:  return 'HIGH',   '🔴'
    elif score < 60: return 'MEDIUM', '🟡'
    else:           return 'LOW',    '🟢'

def predict_student(attendance, prev_marks, study_hours, assignments,
                    internals, extracurricular, parental_edu,
                    internet, health, absences):
    """
    Predict student performance using ensemble of trained models.

    Parameters:
    -----------
    attendance      : float  - Attendance percentage (0-100)
    prev_marks      : float  - Previous semester marks % (0-100)
    study_hours     : float  - Daily study hours (0-12)
    assignments     : float  - Assignment completion % (0-100)
    internals       : float  - Internal marks % (0-100)
    extracurricular : int    - Extracurricular participation (0=No, 1=Yes)
    parental_edu    : int    - Parental education (0=None, 1=Graduate, 2=Postgrad)
    internet        : int    - Internet access (0=No, 1=Yes)
    health          : int    - Health status (1=Poor to 5=Excellent)
    absences        : int    - Number of absent days

    Returns:
    --------
    dict with predicted score, grade, status, risk level, pass probability
    """
    FEATURES = ['attendance','prev_marks','study_hours','assignments',
                'internals','extracurricular','parental_edu','internet',
                'health','absences']

    input_data = pd.DataFrame([{
        'attendance': attendance, 'prev_marks': prev_marks,
        'study_hours': study_hours, 'assignments': assignments,
        'internals': internals, 'extracurricular': extracurricular,
        'parental_edu': parental_edu, 'internet': internet,
        'health': health, 'absences': absences
    }])

    scaled = scaler.transform(input_data[FEATURES])

    # Ensemble regression (average of 3 models)
    s_lr  = float(np.clip(lin_reg.predict(scaled)[0], 0, 100))
    s_rf  = float(np.clip(rf_reg.predict(scaled)[0], 0, 100))
    s_gb  = float(np.clip(gb_reg.predict(scaled)[0], 0, 100))
    avg   = round((s_lr + s_rf + s_gb) / 3, 1)

    # Classification probability
    prob_pass = float(rf_clf.predict_proba(scaled)[0][1]) * 100

    grade  = score_to_grade(avg)
    status = 'PASS' if avg >= 40 else 'FAIL'
    risk, icon = get_risk_level(avg)

    return {
        'predicted_score': avg,
        'score_lr':        round(s_lr, 1),
        'score_rf':        round(s_rf, 1),
        'score_gb':        round(s_gb, 1),
        'grade':           grade,
        'status':          status,
        'risk_level':      risk,
        'risk_icon':       icon,
        'pass_probability': round(prob_pass, 1)
    }

print('✅ predict_student() function ready!')

### 9.1 Test Predictions — Sample Students

In [ ]:
test_students = [
    {'name': 'Arjun (Good Student)',   'attendance':90,'prev_marks':82,'study_hours':6,
     'assignments':88,'internals':78,'extracurricular':1,'parental_edu':2,'internet':1,'health':4,'absences':2},
    {'name': 'Priya (Average Student)','attendance':74,'prev_marks':62,'study_hours':3.5,
     'assignments':68,'internals':58,'extracurricular':0,'parental_edu':1,'internet':1,'health':3,'absences':5},
    {'name': 'Rohit (At Risk)',         'attendance':52,'prev_marks':38,'study_hours':1.5,
     'assignments':42,'internals':35,'extracurricular':0,'parental_edu':0,'internet':0,'health':2,'absences':12},
]

print('='*65)
print(f'{"STUDENT":<25} {"SCORE":>6} {"GRADE":>6} {"STATUS":>6} {"RISK":>7} {"PASS %":>7}')
print('='*65)

for s in test_students:
    r = predict_student(
        s['attendance'], s['prev_marks'], s['study_hours'],
        s['assignments'], s['internals'], s['extracurricular'],
        s['parental_edu'], s['internet'], s['health'], s['absences']
    )
    print(f"{s['name']:<25} {r['predicted_score']:>6} {r['grade']:>6} "
          f"{r['status']:>6} {r['risk_icon']+r['risk_level']:>9} {r['pass_probability']:>6}%")
print('='*65)

In [ ]:
# Visualize test student results
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

risk_colors = {'LOW':'#27ae60','MEDIUM':'#f39c12','HIGH':'#e74c3c'}

for ax, student in zip(axes, test_students):
    result = predict_student(
        student['attendance'], student['prev_marks'], student['study_hours'],
        student['assignments'], student['internals'], student['extracurricular'],
        student['parental_edu'], student['internet'], student['health'], student['absences']
    )
    models  = ['Linear\nReg', 'Random\nForest', 'Grad.\nBoost', 'Ensemble']
    scores  = [result['score_lr'], result['score_rf'], result['score_gb'], result['predicted_score']]
    colors  = ['#5c67f2','#27ae60','#e67e22', risk_colors[result['risk_level']]]
    bars = ax.bar(models, scores, color=colors, edgecolor='white', linewidth=1.5, width=0.55)
    ax.axhline(40, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Pass (40)')
    ax.set_ylim(0, 100); ax.set_ylabel('Score')
    ax.set_title(f"{student['name'].split('(')[0].strip()}\n"
                 f"Score: {result['predicted_score']} | Grade: {result['grade']} | {result['status']}",
                 fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.suptitle('Sample Student Predictions', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 10. Results Summary

| Model | Type | Metric | Score |
|-------|------|--------|-------|
| **Linear Regression** | Regression | R² | **~0.76** |
| Random Forest Reg. | Regression | R² | ~0.69 |
| Gradient Boosting | Regression | R² | ~0.71 |
| **Logistic Regression** | Classification | Accuracy | **~93%** |
| **Random Forest Clf.** | Classification | Accuracy | **~93%** |
| SVM (RBF) | Classification | Accuracy | ~93% |

### Key Takeaways
- **Linear Regression** gives best R² for score prediction — simple and interpretable
- **Random Forest Classifier** is most robust for pass/fail with high precision & recall
- **prev_marks** and **attendance** are the most influential features
- **Ensemble scoring** (average of 3 regressors) gives most stable predictions
- All models saved as `.pkl` files — ready for Streamlit deployment

---
*Student Performance Predictor — Minor Project II | RGPV Bhopal | 2024-25*